# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [ ]:
import pandas as pd
import re

## Variable global
content = ""
sections_content = {}
data_report = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [ ]:
## Global pattern
section_header_pattern = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)

## S1
s1_kv_pattern = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)

##S2
s2_data_pattern = re.compile(
    r"^\s*[MmF]?\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)",
    re.MULTILINE,
)

## S6
s6_header_pattern = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern = re.compile(
    r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)",
    re.MULTILINE,
)

## Extraction des header de chaque section
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [ ]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())
    data_report[section.strip()] = None

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [ ]:
s1_key = sections_titles[0]
matches = s1_kv_pattern.findall(sections_content[s1_key])
data_report[s1_key] = {key.strip(): value.strip() for key, value in matches}
data_report[s1_key]

### S2 - RAPPORT ANALYSE DES PICS

In [ ]:
content = sections_content[sections_titles[1]]


# TODO: Trouver une methode plus robuste pour trouver les titres des colonnes
regex = re.compile(r"^(Numéro[^\n]+)\n(du pic[^\n]+)", re.MULTILINE)
m = re.search(regex, content)
l1 = m.group(1)
l2 = m.group(2)

header = [
    "Numéro du pic",
    "Début (canaux)",
    "Fin (canaux)",
    "Centroïde",
    " Energie (keV)",
    "FWHM (keV)",
    "Surface",
    "Incert.",
    "Fond sous le pic",
]

In [ ]:
# NOTE: Pas obliger de faire ca
# Nettoyage du contenu: 1 - On retire tout les pieds de page
footer_pattern = re.compile(r"^Page\s+\d+\s+sur\s+\d+$", re.MULTILINE)
footers = list(footer_pattern.finditer(content))

for i, f in enumerate(footers):
    start = f.start()
    end = footers[i + 1].start() if i + 1 < len(footers) else len(content)
    content = content[:start] + content[end:]

# Extraction des lignes de données de la section 2
lignes = s2_data_pattern.findall(content)

data_report[sections_titles[1]] = pd.DataFrame(lignes, columns=header)
df = data_report[sections_titles[1]]

# Toutes les colonnes sont numériques
df[header] = df[header].apply(pd.to_numeric, errors="coerce")

df

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

### S5 - RAPPORT LIMITES DE DETECTION

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
def extract_header_s6(sections_content):
    header = []
    str = sections_content[sections_titles[5]]

    lignes = s6_header_pattern.findall(str)

    l1 = re.findall(s6_word_column_pattern, lignes[0][0])
    l2 = re.findall(s6_word_column_pattern, lignes[0][1])
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header


header = extract_header_s6(sections_content)

lignes = re.findall(s6_nucleide_line_pattern, sections_content[sections_titles[5]])

data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"] = pd.DataFrame(
    lignes, columns=header
)
df = data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"]

df[header[1:]] = df[header[1:]].apply(pd.to_numeric, errors="coerce")

# Zone de Test

In [ ]:
import seaborn as sns

sns.pairplot(data_report[sections_titles[1]])